# Lesson 1: Linear regression by hand

Goes with the Lesson 1 page of the Forkcast course. Predict delivery minutes from distance, measure how wrong the prediction is, and improve it with gradient descent that you write yourself.

Runtime: CPU is enough.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. Generate deliveries and look at them

The data is synthetic, made from `minutes = 8 + 3.5 * km + noise`, so later you can compare what the model learned with the truth.

In [ ]:
from forkcast.data import make_deliveries, TRUE_SLOPE, TRUE_INTERCEPT
x, y = make_deliveries(200, seed=0)
print(x.shape, y.shape)

import matplotlib.pyplot as plt
plt.scatter(x, y, s=10); plt.xlabel("km"); plt.ylabel("minutes");

## 3. Write the five functions

Run the cell below to write `forkcast/linear.py`. Replace each `raise NotImplementedError(...)`, in the order the lesson page builds them: `predict` and `mse_loss` first, then `gradients`, then `gradient_step` and `fit`. Re-run the cell every time you change it.

In [ ]:
%%writefile forkcast/linear.py
"""Lesson 1. Linear regression by hand: predict, measure, correct, repeat.

The model is ``minutes = w * distance + b``. ``w`` and ``b`` are 0-dimensional
tensors (single numbers). ``x`` and ``y`` have shape (N,). Replace every
``raise NotImplementedError`` with one to three lines of code.
"""

from __future__ import annotations

import torch
from torch import Tensor


def predict(x: Tensor, w: Tensor, b: Tensor) -> Tensor:
    """The model. (N,) -> (N,): one predicted minute count per delivery."""
    raise NotImplementedError("Lesson 1, step 3: w * x + b")


def mse_loss(pred: Tensor, target: Tensor) -> Tensor:
    """Mean squared error: one number (shape ()) saying how wrong ``pred`` is."""
    raise NotImplementedError("Lesson 1, step 3: square the errors, take the mean")


def gradients(x: Tensor, y: Tensor, w: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    """Slopes of the loss with respect to w and b, each shape ().

    With e = predict(x, w, b) - y:
        dL/dw = 2 * mean(e * x)
        dL/db = 2 * mean(e)
    """
    raise NotImplementedError("Lesson 1, step 4: the two formulas above")


def gradient_step(x: Tensor, y: Tensor, w: Tensor, b: Tensor, lr: float) -> tuple[Tensor, Tensor]:
    """One step downhill: move w and b against their slopes by ``lr`` times the slope."""
    raise NotImplementedError("Lesson 1, step 5: new w = w - lr * dw, same for b")


def fit(x: Tensor, y: Tensor, lr: float = 0.01, steps: int = 3000, w0: float = 0.0, b0: float = 0.0):
    """Run gradient descent from (w0, b0).

    Returns ``(w, b, losses)`` where ``losses`` is a list of Python floats,
    one per step, recorded *before* each step (use ``.item()``).
    """
    raise NotImplementedError("Lesson 1, step 5: loop `steps` times: record loss, take a gradient_step")

### Try a guess by hand

With `w = 1, b = 2` the line is far too shallow. The loss should be about 389, and both slopes should be negative, meaning both knobs need to grow.

In [ ]:
import importlib, torch
from forkcast import linear
importlib.reload(linear)

w, b = torch.tensor(1.0), torch.tensor(2.0)
pred = linear.predict(x, w, b)
print(pred.shape)                                   # torch.Size([200]) -- not (200, 200)!
print(linear.mse_loss(pred, y).item())              # about 389
print(linear.gradients(x, y, w, b))                 # about (-221.6, -36.7)

## 4. Fit, and compare with the truth

In [ ]:
importlib.reload(linear)
w, b, losses = linear.fit(x, y, lr=0.01, steps=3000)
print(f"learned  w={w.item():.3f}  b={b.item():.3f}")
print(f"truth    w={TRUE_SLOPE}  b={TRUE_INTERCEPT}")
print(f"loss went from {losses[0]:.1f} to {losses[-1]:.2f}")

plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (log scale)");

### Break it on purpose

Ten times the learning rate. Watch the loss grow every step until it overflows to `nan`. When a model's loss turns into `nan`, this is the first thing to suspect.

In [ ]:
_, _, bad = linear.fit(x, y, lr=0.1, steps=60)
print([round(v, 1) for v in bad[:5]])
print(bad[-1])

## 5. Definition of done

Expected: `7 passed`, then `14 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_01.py -q

In [ ]:
!pytest -m fast -q

## 6. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 1: linear regression by hand"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main